<a href="https://colab.research.google.com/github/Muchai14/ipo_performance_analysis/blob/main/ipo_performance_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import io
import pandas as pd
import requests

# Question 1: [IPO] Withdrawn IPOs by Company Type
What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?



In [2]:
# 1. Download and load the IPO table
url = "https://www.iposcoop.com/ipos-recently-filed/"
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30,
)
response.raise_for_status()

df = pd.read_html(io.StringIO(response.text))[0]

# Keep withdrawn IPOs with dates before September 11, 2026
df["File Date"] = pd.to_datetime(df["File Date"], errors="coerce")

withdrawn = df.loc[
    df["Expected To Trade"].eq("Withdrawn")
    & df["File Date"].lt("2026-09-11")
].copy()


# 2. Classify companies using the first matching rule
def classify_company(name):
    rules = [
        ("Technologies", ["Technologies"]),
        ("Acquisition Corp", ["Acquisition Corp", "Acquisition Corporation", "Corp"]),
        ("Inc.", ["Inc", "Incorporated"]),
        ("Group", ["Group"]),
        ("Limited", ["Ltd", "Limited"]),
        ("Holdings", ["Holdings", "Holding"]),
    ]

    for company_type, patterns in rules:
        if any(pattern in str(name) for pattern in patterns):
            return company_type

    return "Other"


withdrawn["Company Type"] = withdrawn["Company"].apply(classify_company)


# 3. Parse numeric values; missing values and "-" become NaN
def parse_number(value):
    if pd.isna(value):
        return float("nan")

    cleaned = str(value).replace("$", "").replace(",", "").strip()
    return pd.to_numeric(cleaned, errors="coerce")


withdrawn["Avg_price"] = (
    withdrawn["Price Low"].apply(parse_number)
    + withdrawn["Price High"].apply(parse_number)
) / 2

# 4. Convert shares and estimated offering values to numeric values
for column in ["Shares (millions)", "Est $ Vol (millions)"]:
    withdrawn[column] = withdrawn[column].apply(parse_number)

# 5. Calculate offering value, falling back to the estimate when missing
withdrawn["Shares_offered_value"] = (
    withdrawn["Shares (millions)"] * withdrawn["Avg_price"]
).fillna(withdrawn["Est $ Vol (millions)"])

# 6. Sum offering values by company type, in millions of dollars
totals = (
    withdrawn.groupby("Company Type")["Shares_offered_value"]
    .sum()
    .sort_values(ascending=False)
)

print(f"Withdrawn IPO count: {len(withdrawn)}")
print(totals.to_string())
print(f"\nAnswer: {totals.idxmax()} — ${totals.max():,.3f} million")

Withdrawn IPO count: 31
Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             197.2500
Technologies        184.9000
Group                32.5000

Answer: Acquisition Corp — $499.985 million


# Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)
What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?


In [5]:
%pip install -q yfinance pandas numpy requests lxml

In [8]:
#Download and filter the IPO list
import io
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf

AS_OF = pd.Timestamp("2026-09-11")

# Notebook-compatible output directory
OUTPUT = Path.cwd() / "outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)

yf.set_tz_cache_location(str(OUTPUT / "yf-cache"))

url = "https://www.iposcoop.com/2025-pricings/"
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30,
)
response.raise_for_status()

ipos = pd.read_html(io.StringIO(response.text))[0]

ipos["Offer Date"] = pd.to_datetime(
    ipos["Offer Date"], errors="coerce"
)

ipos["Return_numeric"] = pd.to_numeric(
    ipos["Return"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce",
)

filtered_ipos = ipos.loc[
    ipos["Offer Date"].lt("2025-09-01")
    & ipos["Return_numeric"].notna()
    & ipos["Return_numeric"].ne(0)
].copy()

tickers = (
    filtered_ipos["Symbol"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

print(f"Total IPO rows: {len(ipos)}")
print(f"Filtered IPO rows: {len(filtered_ipos)}")
print(f"Unique tickers: {len(tickers)}")

Total IPO rows: 231
Filtered IPO rows: 146
Unique tickers: 146


In [9]:
#Download daily prices and calculate features
frames = []
failed_tickers = []

for i, ticker in enumerate(tickers, start=1):
    try:
        history = yf.download(
            ticker,
            start="2025-01-01",
            end="2026-09-12",
            interval="1d",
            auto_adjust=True,
            progress=False,
            threads=False,
            timeout=30,
        )

        if history is None or history.empty:
            failed_tickers.append(ticker)
            continue

        # Recent yfinance versions return MultiIndex columns
        if isinstance(history.columns, pd.MultiIndex):
            history.columns = history.columns.get_level_values(0)

        history = (
            history.dropna(subset=["Close"])
            .sort_index()
            .copy()
        )

        if history.empty:
            failed_tickers.append(ticker)
            continue

        history.index = (
            pd.to_datetime(history.index)
            .tz_localize(None)
            .normalize()
        )
        history.index.name = "Date"

        # Calculate features within each ticker's trading history
        history["growth_252d"] = (
            history["Close"] / history["Close"].shift(252)
        )

        history["volatility"] = (
            history["Close"].rolling(30).std() * np.sqrt(252)
        )

        # Use the exact formula specified in the homework
        history["Sharpe"] = (
            history["growth_252d"] - 0.05
        ) / history["volatility"]

        history["Ticker"] = ticker
        frames.append(history.reset_index())

    except Exception as exc:
        failed_tickers.append(ticker)
        print(f"Skipped {ticker}: {exc}")

    finally:
        if i % 20 == 0 or i == len(tickers):
            print(f"Processed {i}/{len(tickers)} tickers")

if not frames:
    raise RuntimeError(
        "No stock data downloaded. Check your connection and retry."
    )

stocks_df = (
    pd.concat(frames, ignore_index=True)
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

print(f"\nSuccessfully downloaded tickers: {stocks_df['Ticker'].nunique()}")
print(f"Unavailable tickers: {failed_tickers}")

# Save the same dataset for Question 3
stocks_df.to_pickle(OUTPUT / "stocks_df.pkl")

Processed 20/146 tickers


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MJID']: YFTzMissingError('possibly delisted; no timezone found')


Processed 40/146 tickers


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['EMPG']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['CAEP']: YFTzMissingError('possibly delisted; no timezone found')


Processed 60/146 tickers


...

Processed 80/146 tickers
Processed 100/146 tickers


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['TBH']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WGRX']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AGH']: YFTzMissingError('possibly delisted; no timezone found')


Processed 120/146 tickers


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['EPWK']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MTSR']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SKBL']: YFTzMissingError('possibly delisted; no timezone found')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MCTR']: YFTzMissingError('possibly delisted; no timezone found')


Processed 140/146 tickers
Processed 146/146 tickers

Successfully downloaded tickers: 132
Unavailable tickers: ['MJID', 'EMPG', 'CAEP', 'PTNM', 'AHL', 'CEPT', 'SDM', 'TBH', 'WGRX', 'AGH', 'EPWK', 'MTSR', 'SKBL', 'MCTR']


In [10]:
#Calculate September 11 statistics and the answer
snapshot = stocks_df.loc[
    stocks_df["Date"].eq(AS_OF)
].copy()

if snapshot.empty:
    raise RuntimeError("No data available for September 11, 2026.")

statistics = snapshot[
    ["growth_252d", "volatility", "Sharpe"]
].describe()

display(statistics)

valid_growth_count = snapshot["growth_252d"].notna().sum()
median_sharpe = snapshot["Sharpe"].median()

print(f"Stocks available on {AS_OF.date()}: {snapshot['Ticker'].nunique()}")
print(f"Stocks with valid 252-day growth: {valid_growth_count}")
print(f"Median growth_252d: {snapshot['growth_252d'].median():.6f}")
print(f"Mean growth_252d: {snapshot['growth_252d'].mean():.6f}")
print(f"\nANSWER — Median Sharpe ratio: {median_sharpe:.6f}")

# Report infinite values without changing the homework's calculation
print(f"Infinite Sharpe values: {np.isinf(snapshot['Sharpe']).sum()}")

Price,growth_252d,volatility,Sharpe
count,130.000000,131.000000,130.000000
mean,1.057613,27.516340,inf
std,3.049007,66.879183,NaN
min,0.001005,0.000000,-0.040147
25%,0.141035,2.301309,0.012042
50%,0.594534,7.885748,0.050142
75%,1.038821,23.452319,0.133619
max,33.638270,627.698106,inf


Stocks available on 2026-09-11: 132
Stocks with valid 252-day growth: 130
Median growth_252d: 0.594534
Mean growth_252d: 1.057613

ANSWER — Median Sharpe ratio: 0.050142
Infinite Sharpe values: 3


In [11]:
#Compare growth and Sharpe rankings
columns = ["Ticker", "growth_252d", "volatility", "Sharpe"]

print("Top 10 stocks by growth:")
display(snapshot.nlargest(10, "growth_252d")[columns])

# Exclude infinite values only for this comparison table
finite_sharpe = snapshot.loc[
    np.isfinite(snapshot["Sharpe"])
]

print("Top 10 stocks by finite Sharpe ratio:")
display(finite_sharpe.nlargest(10, "Sharpe")[columns])

Top 10 stocks by growth:


Price,Ticker,growth_252d,volatility,Sharpe
27337,MGRT,33.638270,131.115959,0.256172
25171,MAMK,6.532663,0.000000,inf
6966,BUUU,5.930931,69.792390,0.084263
24872,MAGH,5.451613,0.000000,inf
2262,ALM,3.480899,36.399039,0.094258
43849,XHLD,2.449658,52.991098,0.045284
34783,RYOJ,2.268293,7.197246,0.308214
13975,EFTY,2.106592,0.000000,inf
36959,SLDE,1.852848,24.497550,0.073593
12364,CV,1.818182,6.754878,0.261764


Top 10 stocks by finite Sharpe ratio:


Price,Ticker,growth_252d,volatility,Sharpe
18640,HCMAU,1.034585,0.182827,5.385326
8253,CEPF,1.024233,0.351583,2.770994
39169,TMDE,0.776166,0.414673,1.751178
38818,TLIH,0.989673,0.648547,1.448889
8674,CEPO,1.031160,0.795258,1.233763
20012,HXHX,0.409565,0.477354,0.753246
16378,FBGL,0.576923,0.838018,0.628773
15618,ETS,1.559252,2.868436,0.526158
11443,CTW,1.164179,2.590424,0.430115
37710,STAK,1.546527,3.907916,0.382948


 # Question 3: [IPO] 'Fixed Months Holding Strategy'
What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?

In [12]:
import pandas as pd

# Sort each ticker's observations chronologically
stocks_df = (
    stocks_df
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
    .copy()
)

# 1. Calculate future growth for holding periods of 1–12 months
growth_columns = []

for month in range(1, 13):
    column = f"future_growth_{month}_m"
    growth_columns.append(column)

    future_close = (
        stocks_df.groupby("Ticker")["Close"]
        .shift(-21 * month)
    )

    stocks_df[column] = future_close / stocks_df["Close"]

# 2. Find the first available trading date for each ticker
min_dates = (
    stocks_df.groupby("Ticker", as_index=False)["Date"]
    .min()
    .rename(columns={"Date": "min_date"})
)

# 3. Inner join to isolate growth from each ticker's first trading date
ipo_entries = min_dates.merge(
    stocks_df,
    left_on=["Ticker", "min_date"],
    right_on=["Ticker", "Date"],
    how="inner",
    validate="one_to_one",
)

# 4. Descriptive statistics for all 12 holding periods
statistics = ipo_entries[growth_columns].describe()
display(statistics)

# Create a readable comparison table
summary = statistics.T[["count", "mean", "50%"]].copy()
summary.columns = ["Stock count", "Mean growth", "Median growth"]
summary.index = pd.Index(range(1, 13), name="Holding months")
summary["Median return (%)"] = (
    summary["Median growth"] - 1
) * 100

display(summary)

# 5. Identify the holding period with the highest median growth
best_month = summary["Median growth"].idxmax()
best_growth = summary.loc[best_month, "Median growth"]

print(f"Optimal holding period: {best_month} month(s)")
print(f"Maximum median growth: {best_growth:.6f}")
print(f"Corresponding median return: {(best_growth - 1) * 100:.2f}%")

,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
count,132.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,130.000000
mean,95.751832,67.979575,51.249160,22.699034,54.244577,64.013104,59.268089,21.460472,20.010294,11.260278,8.904178,5.823784
std,1087.893336,764.388639,573.269360,247.825677,609.321778,720.716187,666.109314,234.733129,216.163486,115.714394,90.034523,55.496952
min,0.072941,0.106197,0.090147,0.063927,0.032245,0.022075,0.013229,0.015130,0.014675,0.011947,0.008681,0.004833
25%,0.710433,0.584863,0.442477,0.392759,0.413434,0.308026,0.245548,0.196227,0.175276,0.174333,0.159666,0.156935
50%,0.935351,0.892958,0.827160,0.730400,0.690909,0.726000,0.662000,0.603508,0.580294,0.533333,0.481811,0.491838
75%,1.114199,1.164977,1.129961,1.197492,1.077133,1.041222,1.018281,1.036588,1.031006,1.034765,1.106045,1.041541
max,12500.000279,8750.000196,6562.500147,2837.500063,6975.000156,8250.000184,7625.000170,2687.500060,2475.000055,1325.000030,1031.250023,633.250012


,Stock count,Mean growth,Median growth,Median return (%)
Holding months,,,,
1,132.0,95.751832,0.935351,-6.464907
2,131.0,67.979575,0.892958,-10.704222
3,131.0,51.249160,0.827160,-17.283953
4,131.0,22.699034,0.730400,-26.959999
5,131.0,54.244577,0.690909,-30.909092
6,131.0,64.013104,0.726000,-27.400002
7,131.0,59.268089,0.662000,-33.800003
8,131.0,21.460472,0.603508,-39.649168
9,131.0,20.010294,0.580294,-41.970590


Optimal holding period: 1 month(s)
Maximum median growth: 0.935351
Corresponding median return: -6.46%


# Question 4: [Strategy] Simple RSI-Based Trading Strategy
What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?



In [14]:
%pip install -q gdown pandas pyarrow

In [15]:
from pathlib import Path

import gdown
import pandas as pd

# 1. Download the dataset
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
file_path = Path("data.parquet")

if not file_path.exists():
    gdown.download(
        f"https://drive.google.com/uc?id={file_id}",
        str(file_path),
        quiet=False,
    )

df = pd.read_parquet(file_path, engine="pyarrow")

print(f"Dataset rows: {len(df):,}")
print("Columns:", df.columns.tolist())

# 2. Identify the date and RSI columns
# Support datasets that store the date in a DatetimeIndex
if isinstance(df.index, pd.DatetimeIndex):
    df = df.reset_index()

date_candidates = [
    column for column in df.columns
    if str(column).lower() in ["date", "datetime", "timestamp"]
]

rsi_candidates = [
    column for column in df.columns
    if str(column).lower() == "rsi"
]

# Also support names such as RSI_14
if not rsi_candidates:
    rsi_candidates = [
        column for column in df.columns
        if str(column).lower().startswith("rsi_")
    ]

if len(date_candidates) != 1:
    raise ValueError(
        f"Specify the date column manually. Candidates: {date_candidates}"
    )

if len(rsi_candidates) != 1:
    raise ValueError(
        f"Specify the RSI column manually. Candidates: {rsi_candidates}"
    )

date_column = date_candidates[0]
rsi_column = rsi_candidates[0]

print(f"Date column: {date_column}")
print(f"RSI column: {rsi_column}")

# 3. Prepare the required fields
dates = (
    pd.to_datetime(df[date_column], errors="coerce", utc=True)
    .dt.tz_localize(None)
    .dt.normalize()
)

rsi = pd.to_numeric(df[rsi_column], errors="coerce")

df["growth_future_30d"] = pd.to_numeric(
    df["growth_future_30d"], errors="coerce"
)

# 4. Select every RSI < 30 signal within the requested date range
signal_mask = (
    dates.between("2000-01-01", "2025-06-01", inclusive="both")
    & rsi.lt(30)
)

selected_df = df.loc[signal_mask].copy()

print(f"\nTotal oversold signals: {len(selected_df):,}")

# Signals without a known forward return cannot be evaluated
missing_returns = selected_df["growth_future_30d"].isna().sum()
selected_df = selected_df.dropna(subset=["growth_future_30d"]).copy()

print(f"Signals missing forward returns: {missing_returns:,}")
print(f"Evaluated trades: {len(selected_df):,}")

if selected_df.empty:
    raise ValueError("No trades with valid forward returns were found.")

# 5. Invest $1,000 independently for every signal
# Consecutive signals for the same ticker count as separate trades
investment_per_trade = 1000

selected_df["return_30d"] = selected_df["growth_future_30d"] - 1
selected_df["profit"] = investment_per_trade * selected_df["return_30d"]

# Exact homework formula
net_income = investment_per_trade * (
    selected_df["growth_future_30d"] - 1
).sum()

# 6. Calculate summary statistics
trade_count = len(selected_df)
total_invested = investment_per_trade * trade_count
average_return = selected_df["return_30d"].mean()
win_rate = selected_df["return_30d"].gt(0).mean()

print("\nStrategy results")
print(f"Number of trades: {trade_count:,}")
print(f"Total invested across all trades: ${total_invested:,.2f}")
print(f"Average 30-day return: {average_return:.2%}")
print(f"Win rate: {win_rate:.2%}")
print(f"Net income: ${net_income:,.2f}")
print(f"\nANSWER — Net income in $ thousands: {net_income / 1000:,.2f}")

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=e3ee18e7-a008-4942-b67b-4acfda6d3067
To: /content/data.parquet
100%|██████████| 130M/130M [00:02<00:00, 47.8MB/s]


Dataset rows: 229,932
Columns: ['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'Year', 'Month', 'Weekday', 'Date', 'growth_1d', 'growth_3d', 'growth_7d', 'growth_30d', 'growth_90d', 'growth_365d', 'growth_future_30d', 'SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'is_positive_growth_30d_future', 'ticker_type', 'index_x', 'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc', 'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext', 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix', 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo', 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk', 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr', 'index_y', 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase', 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine', 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wcl

# Q5. [Exploratory, Optional] Predicting a Positive-Return IPO
Most of the strategies for investing in IPOs deliver negative average and median returns (and even the 75th percentile).

I would buy selectively: favor IPOs with strong revenue growth, reasonable valuations, and positive price momentum after the first month. I would also consider market conditions, diversify positions, and test different exit rules.
To verify improvement, I would test on newer IPOs excluded from strategy development, including trading costs and delisted stocks.